<img src="images/form_factors.png" alt="The AI Maturity Ladder — Five Form Factors of AI Applications" width="900">


# The AI Maturity Ladder: Five Form Factors Of AI Applications

Climb from a plain LLM powered chatbot to an autonomous agent that writes and runs its own code — one rung at a time.

## What You Will Build

A single running example — a support assistant for a fictional product called **Acme Cloud** — implemented five different ways, each one more capable than the last:

1. **The Chatbot** — a stateless LLM that answers from its training knowledge alone.
2. **RAG** — the same LLM, now grounded in Acme Cloud's actual documentation (stored in Oracle AI Database).
3. **The Workflow** — several LLM calls stitched together by *your code* into a reliable pipeline.
4. **The Agent** — an LLM that decides *for itself* which tools to call and when, looping until the job is done.
5. **The Autonomous Agent** — an agent that doesn't just answer, but **writes and runs code** to build durable automation.

Each rung adds exactly one new capability. By the end you'll have a clear mental model of *which* form factor to reach for, and *why*.

## What You Will Learn

- Call Claude for a single-shot response and a multi-turn conversation with the `anthropic` SDK
- Ground a model in your own data with a Retrieval-Augmented Generation (RAG) pipeline on **Oracle AI Database**
- Work through Oracle's **retrieval techniques** — keyword, vector, attribute filtering (pre/in/post), hybrid (RRF), and graph — and shrink vectors with **quantization**
- Orchestrate multiple Claude calls into a deterministic, code-controlled **workflow** (classify → retrieve → draft → review)
- Build a true **agent** with the `claude-agent-sdk` that chooses its own tools
- Let an agent **build automation** — write a script to disk, run it, and fix it autonomously
- Decide which form factor fits a given problem

## The Ladder at a Glance

| # | Form Factor | Who controls the flow? | Data access | Can act on the world? |
|---|-------------|------------------------|-------------|------------------------|
| 1 | **Chatbot** | You (one call) | Training data only | No |
| 2 | **RAG** | You (retrieve → generate) | + Your documents (Oracle) | No |
| 3 | **Workflow** | **Your code** (fixed steps) | + Your documents | Within your pipeline |
| 4 | **Agent** | **The model** (dynamic) | + Tools you provide | Via tools |
| 5 | **Autonomous Agent** | **The model** (dynamic) | + Files & shell | Yes — writes & runs code |

> 💡 **Key Insight — Add one capability at a time:** Each rung of this ladder differs from the one below it by a *single* new power. Naming that difference is the whole point — it tells you the simplest form factor that can solve your problem. Reaching for an autonomous agent when RAG would do is the most common (and expensive) mistake in applied AI.

This taxonomy mirrors Anthropic's guidance in [*Building Effective Agents*](https://www.anthropic.com/research/building-effective-agents): start simple, and add autonomy only when the task genuinely needs it.

## Setup

> **Note:** The cell below installs everything this notebook needs. If your environment already has these packages (for example, a prepared workshop Codespace), you can skip it.
>
> We use **`fastembed`** for embeddings — it runs the model through ONNX (`onnxruntime`) and needs **no PyTorch**, which keeps the notebook portable across platforms and Python versions where a torch wheel may not exist.

In [1]:
# Core SDKs and helpers for every form factor in this notebook.
# - anthropic        -> Form Factors 1-3 (direct Claude API)
# - oracledb         -> Oracle AI Database driver (the RAG store in Form Factor 2)
# - fastembed        -> local, torch-free embeddings for RAG (ONNX); pandas/numpy for vectors & tables
# - claude-agent-sdk -> Form Factors 4-5 (the agent runtime that powers Claude Code)
# - nest_asyncio     -> lets the agent SDK's async code run inside Jupyter

# %pip install -Uq anthropic oracledb fastembed pandas numpy claude-agent-sdk nest_asyncio

## Configure API Access

Every form factor authenticates with a single environment variable, `ANTHROPIC_API_KEY`. Both the `anthropic` SDK and the `claude-agent-sdk` read it automatically. Set it in your shell (or a `.env` you load) before running.

In [1]:
import os
import anthropic

# Both SDKs read ANTHROPIC_API_KEY from the environment automatically.
assert os.environ.get("ANTHROPIC_API_KEY"), (
    "ANTHROPIC_API_KEY is not set — export it (or load a .env) before running this notebook."
)

# One model string, used everywhere. claude-opus-4-8 is the latest, most capable Claude.
MODEL = "claude-opus-4-8"
MAX_TOKENS = 1024  # these are short, demo-sized answers; raise for longer outputs

client = anthropic.Anthropic()
print("Anthropic client ready ✓")

Anthropic client ready ✓


### Smoke-Test the Client

A single round-trip confirms the key works and introduces the shape of every Claude call: `model`, `max_tokens`, an optional `system` prompt, and a list of `messages`.

In [2]:
response = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    system="You are a concise, friendly assistant.",
    messages=[{"role": "user", "content": "In one sentence, what is Claude?"}],
)

# response.content is a list of typed blocks; for a plain answer we read the first text block.
print(response.content[0].text)

Claude is an AI assistant developed by Anthropic, designed to be helpful, harmless, and honest in conversations and tasks.


One tiny helper we'll reuse everywhere: it pulls the plain text out of a response, which can contain several block types (text, thinking, tool-use, …).

In [3]:
def text_of(response) -> str:
    """Concatenate the text blocks of a Claude response into a single string."""
    return "".join(block.text for block in response.content if block.type == "text")


print(text_of(response))

Claude is an AI assistant developed by Anthropic, designed to be helpful, harmless, and honest in conversations and tasks.


# Form Factor 1 — The Chatbot

> 💡 **Key Term — LLM (Large Language Model):** A model trained to predict text. On its own it has no memory between calls, no access to your data, and no ability to take actions. It maps a prompt to a response — nothing more, nothing less.

The simplest useful thing you can build with Claude: send a question, get an answer. No database, no tools, no retrieval. This is where almost every AI project starts.

<img src="images/form_factor_1_stateless.png" alt="Anatomy of a stateless LLM call" width="820">


### A single prompt in, a single response out

`ask_claude` wraps one `messages.create` call. That's the entire chatbot.

In [4]:
def ask_claude(question: str, system: str = "You are a helpful assistant.") -> str:
    """Form Factor 1: a single prompt in, a single response out. Nothing else."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        system=system,
        messages=[{"role": "user", "content": question}],
    )
    return text_of(response)


print(ask_claude("Explain what an API rate limit is, in two sentences."))

An API rate limit is a restriction set by an API provider that caps the number of requests a client can make within a specific time period (such as 100 requests per minute). It helps prevent server overload, ensures fair usage among users, and protects against abuse or malicious attacks.


## 1.1 Adding Memory — by Resending the Conversation

The API is **stateless**: Claude remembers nothing between calls. A "chatbot" feels like it remembers only because *we* resend the full conversation history on every turn.

> 💡 **Key Term — Statelessness:** Each request is independent. To hold a conversation, you accumulate the `messages` list yourself and send all of it every time. The model's "memory" is whatever you put back in front of it.

<img src="images/resending_messages.png" alt="Memory by re-sending the whole conversation" width="820">


The `Chatbot` class below turns the single-shot call into a **multi-turn conversation**. Because the API is stateless, the class keeps its own `history` list and re-sends it on every turn.

- **`send(user_message)`** is the heart of it: it appends the user's message to `history`, calls `client.messages.create(...)` with the *entire* history, extracts the reply, and appends that reply back to `history` before returning it.
- **`self.history.append({...})`** is how "memory" is built. Each turn adds two entries — one `{"role": "user", ...}` and one `{"role": "assistant", ...}` — so the next call sees the whole conversation so far.
- **`system`** is the *system prompt*: a standing instruction that sets the assistant's role and behaviour. It is sent separately from the conversation `messages` and is not part of the back-and-forth turns.
- **`max_tokens`** caps how many tokens Claude may **generate** in its reply — it limits the *output* length, not the input. We set it once as `MAX_TOKENS` and reuse it; raise it when you need longer answers.

### 🧩 TODO 1 — Give the chatbot a memory

The code in the next cell has been removed. Open **[`docs/todo1.md`](docs/todo1.md)** for a guided explanation and the solution snippet, then replace the placeholder.

Run the **`✅ TODO 1 check`** cell right after — it must pass before you continue.

In [ ]:
class Chatbot:
    """A multi-turn chatbot. The 'memory' is just a growing list of messages we resend."""

    def __init__(self, system: str = "You are a helpful assistant."):
        self.system = system
        self.history: list[dict] = []

    def send(self, user_message: str) -> str:
        # 🧩 TODO 1 — make this multi-turn. See docs/todo1.md
        raise NotImplementedError("Complete TODO 1 — open docs/todo1.md")


In [ ]:
# ✅ TODO 1 check — do not edit. This must pass before you continue.
_bot = Chatbot()
_r = _bot.send("Say hello in exactly one word.")
assert isinstance(_r, str) and _r.strip(), "send() should return a non-empty string reply"
assert len(_bot.history) == 2, "history should hold the user message AND the assistant reply"
assert _bot.history[0]["role"] == "user" and _bot.history[1]["role"] == "assistant", \
    "record both sides of the turn, in order (user first, then assistant)"
print("✓ TODO 1 complete — the chatbot now remembers the conversation.")


Below we start a conversation: we instantiate a `Chatbot` and send it our first message. This is **Turn 1** — the bot stores both our message and its reply in `history`, so the next turn has this context to draw on.

In [8]:
bot = Chatbot()
print("Turn 1:", bot.send("Hi! My name is Priya and I run data infrastructure."))


Turn 1: Hi Priya! Nice to meet you.

Data infrastructure is a big, interesting space. What's on your plate these days—are you looking to talk through a specific problem, or just connecting?

A few directions I'm happy to dig into if useful:

- **Architecture decisions** (warehouse vs. lakehouse, batch vs. streaming, build vs. buy)
- **Tooling** (orchestration, ingestion, transformation, catalogs, observability)
- **Scaling/performance** challenges or cost optimization
- **Reliability** (SLAs, data quality, incident response)
- **Team/process** stuff (ownership models, governance, on-call)

What's the context? Let me know your stack or the problem you're chewing on and I can get specific.


In [ ]:
print("\nTurn 2:", bot.send("What's my name and what do I do?"))

> 💡 **Teachable moment:** Turn 2 only works because Turn 1 is still in `bot.history`. Every turn re-sends the whole conversation, so cost and latency grow with the conversation length — which is exactly why later form factors add retrieval and context management instead of just stuffing everything into the history forever.

To chat interactively yourself, define and call the loop below.

In [7]:
def chat_session():
    """Interactive REPL-style chat. Type 'exit' to stop."""
    session = Chatbot()
    print("Chatting with Claude — type 'exit' to quit.\n")
    while True:
        user = input("You: ")
        if user.strip().lower() in {"exit", "quit"}:
            break
        print("Claude:", session.send(user), "\n")


# Uncomment to try it (uses input()):
# chat_session()

## 1.2 The Ceiling of a Chatbot

A chatbot can only draw on what it learned during training. Ask it something specific to *your* world — internal docs, private data, a product it has never seen — and it simply can't know.

<img src="images/form_factor_1_ceiling.png" alt="Where a stateless chatbot hits its limits" width="820">


In [8]:
print(ask_claude(
    "What is the exact API rate limit, in requests per minute, "
    "on Acme Cloud's Pro plan?"
))

I don't have reliable, specific information about the exact API rate limit for Acme Cloud's Pro plan. API rate limits are details that:

- Vary between providers and plans
- Change over time as companies update their offerings
- Sometimes differ based on endpoint, region, or account specifics

I'd also note that "Acme" is often used as a generic placeholder name, so I'm not certain this refers to a specific real product you have in mind.

To get an accurate answer, I'd recommend:

1. **Checking the official documentation** — usually at a URL like `docs.[provider].com` under a "Rate Limits" or "Quotas" section
2. **Looking at your account dashboard** — many providers show your current limits and usage there
3. **Reviewing the pricing/plans page** — rate limits are often listed in plan comparison tables
4. **Contacting their support or sales team** — especially if you need a guaranteed number for capacity planning

If you can point me to specific documentation or paste the relevant text,

> 💡 **Key Insight — The data gap:** Claude has never seen Acme Cloud's documentation, so it can't answer with specifics — at best it explains the gap, at worst it guesses. The fix isn't a bigger model; it's *giving the model the relevant data at question time*. That is exactly what the next rung — RAG — does.

> ### 📌 Key Takeaways — Form Factor 1
> - A chatbot is **one stateless `messages.create` call**: prompt in, text out.
> - "Memory" is an illusion you create by **re-sending the whole conversation** each turn.
> - Its hard ceiling: **no access to your data** and **knowledge frozen at training time** — which motivates every rung above.

# Form Factor 2 — Retrieval-Augmented Generation (RAG)

> 💡 **Key Term — RAG (Retrieval-Augmented Generation):** A pattern where you first *retrieve* the documents relevant to a question, then hand them to the LLM alongside the question. The model answers from real, current data instead of relying on training knowledge alone.

This time we put the knowledge base in a real database — **Oracle AI Database** — store Acme Cloud's documentation as native `VECTOR` embeddings, and work through the retrieval techniques Oracle supports: **keyword**, **vector**, **attribute filtering** (pre / in / post), **hybrid** (keyword + vector), and **graph**. We finish by letting Oracle **quantize** the vectors for scale. Retrieval quality is the single biggest lever on RAG quality, so it is worth seeing the options side by side.

<img src="images/form_factor_2_rag_pipeline.png" alt="Anatomy of a RAG pipeline" width="820">


> 💡 **Key Term — Converged database:** Oracle AI Database stores relational data, full-text indexes, vector embeddings, and graphs together in one engine. That lets us run keyword search, semantic vector search, attribute-filtered and hybrid combinations, and graph traversals — all in SQL, against a single set of tables.
>
> **Prerequisite:** an Oracle AI Database (23ai / 26ai Free) reachable at `localhost:1521/FREEPDB1` with a `VECTOR` user (and the property-graph privilege, used in §2.6.5). Prepared workshop environments provision this; locally you can run the `container-registry.oracle.com/database/free` Docker image.

## 2.1 Connect to Oracle AI Database

A small retry helper, then we keep the connection in `conn` for the rest of the notebook.

In [10]:
import oracledb
import time


def connect_to_oracle(max_retries=3, retry_delay=5):
    """Connect to the local Oracle AI Database, retrying a few times on startup."""
    user = os.environ.get("ORACLE_USER", "VECTOR")
    password = os.environ.get("ORACLE_PASSWORD", "VectorPwd_2025")
    dsn = os.environ.get("ORACLE_DSN", "localhost:1521/FREEPDB1")
    for attempt in range(1, max_retries + 1):
        try:
            print(f"Connection attempt {attempt}/{max_retries}...")
            conn = oracledb.connect(user=user, password=password, dsn=dsn)
            with conn.cursor() as cur:
                cur.execute("SELECT banner FROM v$version WHERE banner LIKE 'Oracle%'")
                print("Connected:", cur.fetchone()[0])
            return conn
        except oracledb.OperationalError as e:
            print(f"Attempt {attempt} failed: {e}")
            if attempt < max_retries:
                time.sleep(retry_delay)
            else:
                raise


conn = connect_to_oracle()

Connection attempt 1/3...
Connected: Oracle AI Database 26ai Free Release 23.26.0.0.0 - Develop, Learn, and Run for Free


## 2.2 The Knowledge Base

Acme Cloud's documentation — a dozen short docs across categories (billing, API, security, data, …). Each becomes one row in Oracle; the `category` field will drive both **attribute filtering** (§2.6.3) and **graph retrieval** (§2.6.5).

In [11]:
DOCS = [
    {"doc_id": "plans", "title": "Plans & Pricing", "category": "billing",
     "content": "Acme Cloud offers three plans. Free includes 1 project and community support. "
                "Pro is $49 per user per month with email support and advanced analytics. "
                "Enterprise has custom pricing, SSO, and a dedicated support engineer."},
    {"doc_id": "rate_limits", "title": "API Rate Limits", "category": "api",
     "content": "Acme Cloud enforces API rate limits per plan. The Free plan allows 60 requests "
                "per minute. The Pro plan allows 1,000 requests per minute. Enterprise rate limits "
                "are negotiated per contract."},
    {"doc_id": "upgrade", "title": "Upgrading Your Plan", "category": "billing",
     "content": "To upgrade, open Settings, then Billing, then Change Plan. Upgrades take effect "
                "immediately and are pro-rated for the current billing cycle."},
    {"doc_id": "regions", "title": "Data Residency & Regions", "category": "data",
     "content": "Acme Cloud stores data in US, EU (Frankfurt), and APAC (Singapore) regions. "
                "The region is chosen at project creation and cannot be changed afterward."},
    {"doc_id": "sla", "title": "Service Level Agreement", "category": "reliability",
     "content": "Acme Cloud guarantees 99.9% uptime on the Pro plan and 99.99% on Enterprise. "
                "SLA credits are issued automatically when monthly uptime falls below target."},
    {"doc_id": "support", "title": "Support Channels & Response Times", "category": "support",
     "content": "Free plan customers use the community forum. Pro email support responds within one "
                "business day. Enterprise includes 24/7 support with a one-hour response target for "
                "critical issues."},
    {"doc_id": "api_keys", "title": "Creating & Rotating API Keys", "category": "api",
     "content": "Create and rotate API keys under Settings, then API Keys. Rotating a key immediately "
                "revokes the previous one, so update your applications before rotating."},
    {"doc_id": "sso", "title": "Single Sign-On (SSO)", "category": "security",
     "content": "SSO is available on the Enterprise plan and supports SAML 2.0 and OIDC. An "
                "administrator configures the identity provider under Settings, then Security, then SSO."},
    {"doc_id": "webhooks", "title": "Webhooks", "category": "integrations",
     "content": "Acme Cloud can send webhooks on project events. Configure endpoint URLs under "
                "Settings, then Webhooks. Failed deliveries are retried with exponential backoff for "
                "up to 24 hours."},
    {"doc_id": "backups", "title": "Backups & Recovery", "category": "data",
     "content": "Acme Cloud takes automated daily backups retained for 30 days on Pro and 90 days on "
                "Enterprise. Point-in-time recovery is available on the Enterprise plan."},
    {"doc_id": "roles", "title": "Team Roles & Permissions", "category": "account",
     "content": "Acme Cloud supports Owner, Admin, Member, and Viewer roles. Only Owners and Admins "
                "can manage billing, invite teammates, or rotate API keys."},
    {"doc_id": "export", "title": "Exporting Your Data", "category": "data",
     "content": "You can export project data as JSON or CSV from Settings, then Export. Large exports "
                "are emailed as a downloadable archive when ready."},
]
print(len(DOCS), "documents in the knowledge base.")

12 documents in the knowledge base.


## 2.3 Embed the Documents

> 💡 **Key Term — Embedding:** A numeric vector that captures the meaning of text. Texts with similar meaning have similar vectors, which lets us search by *meaning* rather than exact keywords.

We use `nomic-embed-text-v1.5` via **fastembed** (ONNX — no PyTorch). It's an *asymmetric* model: documents and queries are embedded with different prefixes, which fastembed applies for us — `embed()` for documents, `query_embed()` for queries.

> **Note:** The first call downloads the model (a few hundred MB) and may take a minute.

**Why normalize?** The `_unit` helper divides each embedding by its length (its L2 norm) so every vector has magnitude 1. Two things follow:

- **Cosine similarity becomes a plain dot product.** Cosine measures the *angle* between two vectors and ignores their length; once vectors are unit-length, their dot product *is* the cosine. That keeps the similarity math simple and consistent.
- **Every vector sits on the same scale.** Comparisons aren't skewed by one document happening to produce a longer raw vector than another — only *direction* (meaning) matters.

Oracle's `VECTOR_DISTANCE(..., COSINE)` normalizes internally too, so this mainly keeps our own client-side math — the graph similarity edges in §2.6.5 — clean and unambiguous.

In [ ]:
from fastembed import TextEmbedding
import numpy as np

embedder = TextEmbedding(model_name="nomic-ai/nomic-embed-text-v1.5")


def _unit(v):
    """Normalize to a unit vector so cosine similarity is a plain dot product."""
    v = np.asarray(v, dtype=np.float32)
    n = np.linalg.norm(v)
    return v / n if n else v


/Users/richmondalake/opt/anaconda3/envs/dbtlabs/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 5 files: 100%|██████████| 5/5 [01:17<00:00, 15.44s/it]


Embedded 12 docs at dimension 768.


**How `doc_texts` is built.** For each document we concatenate two fields — the **title** and the **content** — into a single string with `f"{d['title']}. {d['content']}"`, then embed that one string to get one vector per document.

**Why concatenate several fields?** An embedding turns *one* piece of text into *one* vector, so everything we want a document to be findable by has to live in that text:

- The **title** carries dense, high-signal keywords ("API Rate Limits", "Single Sign-On") that pull the vector toward the document's topic.
- The **content** adds the detail and surrounding context that lets semantic, paraphrased queries match.

Joining them with `". "` gives the model a clean, sentence-like input and a richer representation than either field alone — so a query can match on the title's keywords *or* the body's meaning. Fields we only filter or display on (like `category`) are kept as ordinary columns instead of being folded into the embedding.

In [ ]:
doc_texts = [f"{d['title']}. {d['content']}" for d in DOCS]
doc_vectors = np.array([_unit(v) for v in embedder.embed(doc_texts)], dtype=np.float32)
dim = int(doc_vectors.shape[1])
print(f"Embedded {len(DOCS)} docs at dimension {dim}.")

## 2.4 Create the Table

The table stores the document plus a native `VECTOR(dim, FLOAT32)` column. The DDL drops any prior copy first so the cell is safe to re-run.

> ⚠️ **The `VECTOR` dimension must match your embedding model.** The column below is declared `VECTOR(dim, FLOAT32)`, where `dim` is the value we read from the model in §2.3 (`768` for `nomic-embed-text-v1.5`). The number in the column definition has to equal the length of the vectors you insert — declare `VECTOR(512, ...)` while the model emits 768-dim vectors and **every insert fails** with a dimension-mismatch error. We thread `dim` through programmatically precisely so the table always matches whatever model produced the embeddings.

In [15]:
import array

table_ddl = f"""
BEGIN EXECUTE IMMEDIATE 'DROP TABLE acme_docs CASCADE CONSTRAINTS PURGE';
EXCEPTION WHEN OTHERS THEN IF SQLCODE != -942 THEN RAISE; END IF; END;
/
CREATE TABLE acme_docs (
    doc_id    VARCHAR2(64) PRIMARY KEY,
    title     VARCHAR2(400),
    category  VARCHAR2(64),
    content   VARCHAR2(4000),
    embedding VECTOR({dim}, FLOAT32)
)
"""
with conn.cursor() as cur:
    for stmt in table_ddl.split("/"):
        if stmt.strip():
            cur.execute(stmt)
conn.commit()
print(f"Table ACME_DOCS created (VECTOR dim={dim}).")

Table ACME_DOCS created (VECTOR dim=768).


### Add the indexes

Two indexes power the retrieval techniques that follow:

| Index | Type | Powers |
|---|---|---|
| `acme_text_idx` | Oracle Text (`CTXSYS.CONTEXT`) | keyword & hybrid search (`CONTAINS`) |
| `acme_vec_idx` | Vector (HNSW) | semantic search + attribute filtering (the `VECTOR_INDEX_TRANSFORM` hint in §2.6.3 needs an HNSW index) |

> 💡 **Key Term — HNSW index:** A graph-based vector index that turns similarity search from a full scan into a sub-second lookup. On our tiny table an exact scan is already instant, so the vector index is mainly here to (a) demonstrate the pattern and (b) enable the pre/in/post-filter hints. We tolerate it being skipped if the database's vector pool isn't configured.

<img src="images/hnsw_index.png" alt="How an HNSW index finds nearest neighbours" width="820">


In [16]:
with conn.cursor() as cur:
    # Full-text index — REQUIRED for keyword (CONTAINS) and hybrid search.
    try:
        cur.execute("DROP INDEX acme_text_idx")
    except oracledb.DatabaseError:
        pass
    cur.execute("""
        CREATE INDEX acme_text_idx ON acme_docs(content)
        INDEXTYPE IS CTXSYS.CONTEXT PARAMETERS ('SYNC (ON COMMIT)')
    """)
    print("Oracle Text index created (keyword search ready).")

    # HNSW vector index — enables APPROX search and the pre/in/post-filter hints.
    try:
        cur.execute("DROP INDEX acme_vec_idx")
    except oracledb.DatabaseError:
        pass
    try:
        cur.execute("""
            CREATE VECTOR INDEX acme_vec_idx ON acme_docs(embedding)
            ORGANIZATION INMEMORY NEIGHBOR GRAPH
            DISTANCE COSINE WITH TARGET ACCURACY 90
            PARAMETERS (TYPE HNSW, NEIGHBORS 16, EFCONSTRUCTION 200)
        """)
        print("HNSW vector index created.")
    except oracledb.DatabaseError as e:
        print("HNSW index skipped (exact search still works):", str(e).splitlines()[0][:90])
conn.commit()

Oracle Text index created (keyword search ready).
HNSW vector index created.


## 2.5 Ingest the Documents

Each embedding is sent as a Python `array('f', ...)`, which Oracle's driver maps straight onto the `VECTOR` column. `executemany` inserts the whole batch in one round trip.

In [ ]:
rows = [
    (d["doc_id"], d["title"], d["category"], d["content"], array.array("f", vec))
    for d, vec in zip(DOCS, doc_vectors.astype(np.float32).tolist())
]

with conn.cursor() as cur:
    cur.executemany(
        "INSERT INTO acme_docs (doc_id, title, category, content, embedding) "
        "VALUES (:1, :2, :3, :4, :5)",
        rows,
    )
conn.commit()

Rows in ACME_DOCS: 12


After inserting, we run a quick `SELECT COUNT(*)` as a **sanity check** — confirming all twelve rows actually landed in `ACME_DOCS` before we start querying. It is a cheap guard against a partial or failed ingest, so a later "no results" surprise is never just missing data.

In [18]:

with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM acme_docs")
    print("Rows in ACME_DOCS:", cur.fetchone()[0])

Rows in ACME_DOCS: 12


## 2.6 Retrieval Techniques

> 💡 **Key Insight — No single retrieval strategy wins everywhere.** Keyword search nails exact terms but misses synonyms; vector search captures meaning but can drift off-topic; attribute filtering narrows by structured metadata; hybrid methods combine keyword + vector; graph retrieval follows relationships. We'll work through each on the same data.

First, three small helpers shared by the techniques below.

<img src="images/form_factor_2_retrieval_techniques.png" alt="Five ways to retrieve from Oracle" width="820">


The next cell defines four small helpers used by every technique that follows:

| Helper | What it does |
|---|---|
| `_query_vec(text)` | Embeds a query string into a unit vector (fastembed adds nomic's query-side prefix automatically). |
| `embed_query(text)` | Wraps `_query_vec` and returns the vector as a Python `array('f')` — the form Oracle's driver binds to a `:q` `VECTOR` parameter. |
| `contains_arg(phrase)` | Makes a phrase safe for Oracle Text `CONTAINS` by quoting multi-word phrases. |
| `show(rows, columns)` | Renders a `(rows, columns)` query result as a pandas DataFrame for readable output. |

In [20]:
import pandas as pd


def _query_vec(text: str):
    """Embed a query as a unit vector (fastembed applies nomic's query prefix internally)."""
    return _unit(next(embedder.query_embed(text)))


def embed_query(text: str):
    """A query vector in Oracle's expected array form (FLOAT32)."""
    return array.array("f", _query_vec(text).astype(np.float32).tolist())


def contains_arg(phrase: str) -> str:
    """Make a phrase safe for Oracle Text CONTAINS (wrap multi-word phrases in quotes)."""
    return f'"{phrase}"' if " " in phrase.strip() else phrase


def show(rows, columns):
    """Render result rows as a DataFrame for easy reading."""
    return pd.DataFrame(rows, columns=columns)

### 2.6.1 Keyword Search (Oracle Text)

Full-text search over the `content` column. `SCORE(1)` ranks by term relevance. Great for exact terms — but it only matches words that literally appear.

**How keyword search works in Oracle AI Database.** The `CONTAINS(content, ...)` predicate is powered by the **Oracle Text** index we built (`CTXSYS.CONTEXT`). At index time Oracle *tokenizes* every document — splitting text into words, lower-casing them, and recording which documents each term appears in (an **inverted index**). At query time it looks the search term up in that index instead of scanning every row, so full-text search stays fast even over large tables.

**What `SCORE(1)` means.** When a row matches, Oracle Text assigns it a **relevance score** (the `1` ties the score to this particular `CONTAINS` call). The score is higher when the term is more prominent in a document — driven by how often it occurs relative to the document's length — and is returned on a **0–100** scale. Ordering by `SCORE(1) DESC` therefore puts the most on-topic documents first.

In [ ]:
def keyword_search(query: str, k: int = 5):
    sql = f"""
        SELECT doc_id, title, category, SCORE(1) AS score
        FROM acme_docs
        WHERE CONTAINS(content, :kw, 1) > 0
        ORDER BY SCORE(1) DESC
        FETCH FIRST {int(k)} ROWS ONLY
    """
    with conn.cursor() as cur:
        cur.execute(sql, kw=contains_arg(query))
        return cur.fetchall(), [d[0] for d in cur.description]

,DOC_ID,TITLE,CATEGORY,SCORE
0,sso,Single Sign-On (SSO),security,11
1,plans,Plans & Pricing,billing,5


In [22]:
rows, cols = keyword_search("SSO")
show(rows, cols)

,DOC_ID,TITLE,CATEGORY,SCORE
0,sso,Single Sign-On (SSO),security,11
1,plans,Plans & Pricing,billing,5


### 2.6.2 Vector Search (semantic)

Embed the query, then rank rows by cosine similarity with `VECTOR_DISTANCE`. This matches *meaning*, so a paraphrase with none of the document's exact words still finds the right doc.

> 💡 **Teachable moment:** The query below contains none of the words "rate", "limit", or "API" — yet vector search returns the **API Rate Limits** doc first. That semantic recall is what keyword search structurally cannot do.

<img src="images/vector_search_embedding_space.png" alt="Vector search retrieves by meaning" width="820">


**How vector search works.** Three steps: **(1)** embed the query with the *same* model used for the documents, so query and documents share one vector space; **(2)** compute the **cosine distance** between the query vector and every stored `embedding` with `VECTOR_DISTANCE(embedding, :q, COSINE)`; **(3)** return the closest rows. Because "closeness" in that shared space means "similar in meaning," a paraphrase with none of the document's exact words still lands near the right document.

**What the score means.** `VECTOR_DISTANCE(..., COSINE)` returns a *distance* (0 = identical direction, larger = further apart). We report `1 - distance` as a **similarity** — for our unit-length embeddings that lands in roughly `[0, 1]`, where **higher = more similar** — which is why we `ORDER BY similarity DESC`. On a large table you'd add an HNSW index and use `FETCH APPROX` to make this an *approximate* nearest-neighbour search (sub-linear) instead of scanning every row.

### 🧩 TODO 2 — Semantic search with Oracle vectors

The code in the next cell has been removed. Open **[`docs/todo2.md`](docs/todo2.md)** for a guided explanation and the solution snippet, then replace the placeholder.

Run the **`✅ TODO 2 check`** cell right after — it must pass before you continue.

In [ ]:
def vector_search(query: str, k: int = 5):
    # 🧩 TODO 2 — semantic (vector) search in Oracle AI Database. See docs/todo2.md
    raise NotImplementedError("Complete TODO 2 — open docs/todo2.md")


In [ ]:
# ✅ TODO 2 check — do not edit.
_rows, _cols = vector_search("What is the Pro plan API rate limit?", k=3)
assert _rows, "vector_search should return some rows"
assert _rows[0][0] == "rate_limits", f"the closest doc should be 'rate_limits', got {_rows[0][0]!r}"
print("✓ TODO 2 complete — vector search ranks the right document first.")


In [24]:
rows, cols = vector_search("how many requests can I send before being throttled?")
show(rows, cols)

,DOC_ID,TITLE,CATEGORY,SIMILARITY
0,rate_limits,API Rate Limits,api,0.6320
1,webhooks,Webhooks,integrations,0.5756
2,support,Support Channels & Response Times,support,0.5268
3,sla,Service Level Agreement,reliability,0.4910
4,upgrade,Upgrading Your Plan,billing,0.4897


### 2.6.3 Attribute Filtering — Pre / In / Post

Real queries usually combine semantic similarity with a **structured filter** ("only `data` docs", "only this tenant", "only last 30 days"). Oracle lets you control *when* that `WHERE` predicate is applied relative to the HNSW graph traversal, via the `VECTOR_INDEX_TRANSFORM` optimizer hint:

| Mode | Hint | What happens | Trade-off |
|---|---|---|---|
| **Pre-filter** | `pre_filter_with_join_back` | Apply the `WHERE` filter **first**, then rank the surviving rows by vector distance | Guarantees you get *topN within the filtered set*; can be costly if the filter is broad |
| **In-filter** | `in_filter_with_join_back` | The filter is checked **during** the graph traversal — only matching nodes are explored | Often the best balance; available on HNSW indexes |
| **Post-filter** | `post_filter_without_join_back` | Rank by vector distance **first**, then drop rows that fail the filter | Fastest, but can **under-return** — if most top-K vectors fail the filter, you get fewer than `k` rows |

> 💡 **Key Term — pre / in / post filtering:** the same query, the same answer set *in principle* — but the order of "filter" vs "search" changes both performance and recall. Omit the hint and Oracle's optimizer picks for you.

<img src="images/attribute_filtering.png" alt="Pre, in, and post filtering change your results" width="820">


The `filtered_search` function below runs a vector search **restricted to one `category`**, and lets us choose the filtering strategy. Three pieces of its SQL are worth reading closely:

- **`/*+ vector_index_transform(acme_docs acme_vec_idx <filtertype>) */`** — an Oracle *optimizer hint*. It tells the planner to use our HNSW index `acme_vec_idx` on `acme_docs` and to apply the `WHERE category = :cat` predicate using the chosen strategy (`pre_filter_with_join_back`, `in_filter_with_join_back`, or `post_filter_without_join_back`). Hints are advisory — remove it and Oracle picks a strategy itself.
- **`VECTOR_DISTANCE(embedding, :q, COSINE)`** — computes the cosine distance between each row's stored `embedding` and the bound query vector `:q`. We `ORDER BY` it (closest first) and also report `1 - distance` as a similarity.
- **`FETCH APPROX FIRST k ROWS ONLY WITH TARGET ACCURACY 90`** — `APPROX` requests *approximate* nearest-neighbour search through the HNSW index instead of an exact full scan, and `TARGET ACCURACY 90` says "aim to return about 90% of the true top-k." That accuracy figure is the classic ANN trade-off: lower is faster and cheaper, higher is slower but closer to exact.

The cell defines the function and runs it once with the default pre-filter; the **next** cell then compares all three strategies side by side.

In [25]:
def filtered_search(query: str, category: str, k: int = 5,
                    filtertype: str = "pre_filter_with_join_back"):
    """Vector search restricted to a category, with the chosen pre/in/post filter strategy."""

    q = embed_query(query)

    sql = f"""
        SELECT /*+ vector_index_transform(acme_docs acme_vec_idx {filtertype}) */
               doc_id, title, category,
               ROUND(1 - VECTOR_DISTANCE(embedding, :q, COSINE), 4) AS similarity
        FROM acme_docs
        WHERE category = :cat
        ORDER BY VECTOR_DISTANCE(embedding, :q, COSINE)
        FETCH APPROX FIRST {int(k)} ROWS ONLY WITH TARGET ACCURACY 90
    """

    with conn.cursor() as cur:
        cur.execute(sql, q=q, cat=category)
        return cur.fetchall(), [d[0] for d in cur.description]

In [26]:

rows, cols = filtered_search("keep my data safe and recoverable", category="data")
show(rows, cols)

,DOC_ID,TITLE,CATEGORY,SIMILARITY
0,backups,Backups & Recovery,data,0.6402
1,export,Exporting Your Data,data,0.5365
2,regions,Data Residency & Regions,data,0.5288


**Compare the three strategies** on the same query and filter. On a 12-row table they return the *same* rows — the difference is *how* Oracle gets there (and, at scale, whether post-filter under-returns). Watch the order of operations, not the rows.

In [27]:
query, category = "keep my data safe and recoverable", "data"
print(f"query={query!r}  filter: category='{category}'\n")
for ft in ["pre_filter_with_join_back", "in_filter_with_join_back", "post_filter_without_join_back"]:
    rows, cols = filtered_search(query, category, k=5, filtertype=ft)
    print(f"{ft:32s} -> {[r[0] for r in rows]}")

query='keep my data safe and recoverable'  filter: category='data'

pre_filter_with_join_back        -> ['backups', 'export', 'regions']
in_filter_with_join_back         -> ['backups', 'export', 'regions']
post_filter_without_join_back    -> ['backups', 'export', 'regions']


> 💡 **Teachable moment — why post-filter can under-return:** Imagine 1M docs and you ask for the top 10 by vector, then keep only `category='data'`. If none of the global top 10 happen to be `data`, you get **zero** rows back. Pre-filter and in-filter avoid that by constraining to `data` *before/while* ranking — so they always fill `k` if enough matches exist. That's the practical reason to prefer pre/in-filter when the filter is selective.

### 2.6.4 Hybrid Search — Keyword + Vector (RRF)

> 💡 **Key Term — RRF (Reciprocal Rank Fusion):** Merge two independent ranked lists — one from vector search, one from keyword search — using each item's *rank position*, not its raw score: `score = 1/(k + rank_vector) + 1/(k + rank_keyword)`. It normalizes across incompatible scoring scales and needs no weight tuning.

<img src="images/hybrid_search_rrf.png" alt="Reciprocal Rank Fusion merges two rankings" width="820">


**How is this different from the filtering we just did?** Attribute filtering (§2.6.3) works from *one* signal — vector similarity — and uses a structured `WHERE` predicate to decide *which rows are even eligible* (e.g. only `category='data'`). It **narrows** a single search.

RRF does something different: it runs **two independent searches** — keyword and vector — each producing its own ranked list, then **fuses** them. Nothing is filtered out; instead, a document that ranks well in *either* list (or modestly in both) rises to the top. So filtering is about *eligibility on an attribute*, while RRF is about *combining two retrieval signals* — catching exact-term matches and semantic matches in one result set.

In [33]:
def rrf_search(query: str, k: int = 5, per_list: int = 10, rrf_k: int = 60):
    q = embed_query(query)
    sql = f"""
        WITH
        vec AS (
            SELECT doc_id, title,
                   ROW_NUMBER() OVER (ORDER BY VECTOR_DISTANCE(embedding, :q, COSINE)) AS r_vec
            FROM acme_docs
            ORDER BY VECTOR_DISTANCE(embedding, :q, COSINE)
            FETCH FIRST {int(per_list)} ROWS ONLY
        ),
        txt AS (
            SELECT doc_id, title,
                   ROW_NUMBER() OVER (ORDER BY SCORE(1) DESC) AS r_txt
            FROM acme_docs
            WHERE CONTAINS(content, :kw, 1) > 0
            ORDER BY SCORE(1) DESC
            FETCH FIRST {int(per_list)} ROWS ONLY
        ),
        fused AS (
            SELECT COALESCE(v.doc_id, t.doc_id) AS doc_id,
                   COALESCE(v.title, t.title) AS title,
                   NVL(v.r_vec, 999999) AS r_vec,
                   NVL(t.r_txt, 999999) AS r_txt
            FROM vec v FULL OUTER JOIN txt t ON t.doc_id = v.doc_id
        )
        SELECT doc_id, title, r_vec, r_txt,
               ROUND(1.0/(:rk + r_vec) + 1.0/(:rk + r_txt), 6) AS rrf_score
        FROM fused
        ORDER BY rrf_score DESC, title
        FETCH FIRST {int(k)} ROWS ONLY
    """
    with conn.cursor() as cur:
        cur.execute(sql, q=q, kw=contains_arg(query), rk=rrf_k)
        return cur.fetchall(), [d[0] for d in cur.description]


rows, cols = rrf_search("Enterprise")
show(rows, cols)

,DOC_ID,TITLE,R_VEC,R_TXT,RRF_SCORE
0,sso,Single Sign-On (SSO),1,2,0.032522
1,backups,Backups & Recovery,3,1,0.032266
2,support,Support Channels & Response Times,2,4,0.031754
3,sla,Service Level Agreement,6,3,0.031025
4,rate_limits,API Rate Limits,4,6,0.030777


> 💡 **The native, turnkey way — Hybrid Vector Index:** Oracle 23ai/26ai can combine full-text and semantic search in a *single* index and query it with one call:
>
> ```sql
> CREATE HYBRID VECTOR INDEX acme_hybrid_idx ON acme_docs(content)
>   PARAMETERS ('MODEL my_indb_model');
>
> SELECT DBMS_HYBRID_VECTOR.SEARCH(json('{
>   "hybrid_index_name": "acme_hybrid_idx",
>   "vector": { "search_text": "rate limits" },
>   "text":   { "contains": "Pro" },
>   "return": { "topN": 5 } }')) FROM dual;
> ```
>
> It requires an **in-database ONNX embedding model** (it chunks and embeds the documents itself), so it's the production path for large document corpora. Here we use client-side embeddings and explicit SQL so each retrieval step is visible, fusing keyword + vector with RRF above.

### 2.6.5 Graph Retrieval (SQL Property Graph)

> 💡 **Key Term — Graph retrieval:** Seed the search with vector similarity, then **traverse relationship edges** to pull in docs the vector search alone would miss. We build two edge types:
> - `SIMILAR_TO` — each doc's nearest neighbours by embedding (computed once, stored as edges)
> - `IN_CATEGORY` — links every doc to its category vertex, so two docs in the same category are two hops apart
>
> The query uses Oracle's SQL property graph with the SQL/PGQ `GRAPH_TABLE ... MATCH` operator — the current standard for graph queries in Oracle AI Database.
>
> **Prerequisite:** the `CREATE PROPERTY GRAPH` privilege (granted in prepared workshop environments).

We build this in four small steps: 

(1) create the edge tables, 

(2) compute and load the edges, 

(3) register the property graph, 

(4) query it.

<img src="images/graph_retrieval.png" alt="GraphRAG retrieves by following relationships" width="820">


**Step 1 — create the relational edge tables** that will back the graph.

In [34]:
graph_ddl = """
BEGIN EXECUTE IMMEDIATE 'DROP TABLE doc_similarities'; EXCEPTION WHEN OTHERS THEN IF SQLCODE != -942 THEN RAISE; END IF; END;
/
BEGIN EXECUTE IMMEDIATE 'DROP TABLE doc_categories'; EXCEPTION WHEN OTHERS THEN IF SQLCODE != -942 THEN RAISE; END IF; END;
/
BEGIN EXECUTE IMMEDIATE 'DROP TABLE categories'; EXCEPTION WHEN OTHERS THEN IF SQLCODE != -942 THEN RAISE; END IF; END;
/
CREATE TABLE categories (category VARCHAR2(64) PRIMARY KEY)
/
CREATE TABLE doc_categories (
    doc_id   VARCHAR2(64) NOT NULL,
    category VARCHAR2(64) NOT NULL,
    CONSTRAINT pk_doc_categories PRIMARY KEY (doc_id, category),
    CONSTRAINT fk_dc_doc FOREIGN KEY (doc_id) REFERENCES acme_docs(doc_id),
    CONSTRAINT fk_dc_cat FOREIGN KEY (category) REFERENCES categories(category)
)
/
CREATE TABLE doc_similarities (
    source_doc_id VARCHAR2(64) NOT NULL,
    target_doc_id VARCHAR2(64) NOT NULL,
    sim_score NUMBER(8,6) NOT NULL,
    rank_no   NUMBER(5) NOT NULL,
    CONSTRAINT pk_doc_similarities PRIMARY KEY (source_doc_id, target_doc_id),
    CONSTRAINT fk_ds_src FOREIGN KEY (source_doc_id) REFERENCES acme_docs(doc_id),
    CONSTRAINT fk_ds_tgt FOREIGN KEY (target_doc_id) REFERENCES acme_docs(doc_id),
    CONSTRAINT ck_ds_not_self CHECK (source_doc_id <> target_doc_id)
)
"""
with conn.cursor() as cur:
    for stmt in graph_ddl.split("/"):
        if stmt.strip():
            cur.execute(stmt)
conn.commit()
print("Edge tables created: categories, doc_categories, doc_similarities.")

Edge tables created: categories, doc_categories, doc_similarities.


**Step 2 — compute and load the edges.** Category edges come straight from each doc's `category`; similarity edges are each doc's top-3 nearest neighbours by cosine, computed once in NumPy.

In [35]:
doc_ids = [d["doc_id"] for d in DOCS]
categories = sorted({d["category"] for d in DOCS})
doc_cat_rows = [(d["doc_id"], d["category"]) for d in DOCS]

sim = doc_vectors @ doc_vectors.T          # cosine similarity (vectors are normalized)
np.fill_diagonal(sim, -np.inf)             # never link a doc to itself
TOP_NEIGHBORS = 3
sim_rows = []
for i, src in enumerate(doc_ids):
    for rank, j in enumerate(np.argsort(sim[i])[::-1][:TOP_NEIGHBORS], start=1):
        sim_rows.append((src, doc_ids[j], round(float(sim[i][j]), 6), rank))

with conn.cursor() as cur:
    cur.executemany("INSERT INTO categories (category) VALUES (:1)", [(c,) for c in categories])
    cur.executemany("INSERT INTO doc_categories (doc_id, category) VALUES (:1, :2)", doc_cat_rows)
    cur.executemany(
        "INSERT INTO doc_similarities (source_doc_id, target_doc_id, sim_score, rank_no) "
        "VALUES (:1, :2, :3, :4)",
        sim_rows,
    )
conn.commit()
print(f"Loaded {len(categories)} categories, {len(doc_cat_rows)} IN_CATEGORY edges, "
      f"{len(sim_rows)} SIMILAR_TO edges.")

Loaded 8 categories, 12 IN_CATEGORY edges, 36 SIMILAR_TO edges.


**Step 3 — register the SQL property graph** over the docs (vertices) and the two edge tables.

In [36]:
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM user_property_graphs WHERE graph_name = 'ACME_GRAPH'")
    if cur.fetchone()[0] > 0:
        cur.execute("DROP PROPERTY GRAPH acme_graph")
    cur.execute("""
        CREATE PROPERTY GRAPH acme_graph
        VERTEX TABLES (
            acme_docs  KEY (doc_id)   LABEL doc      PROPERTIES (doc_id, title, category),
            categories KEY (category) LABEL category PROPERTIES (category)
        )
        EDGE TABLES (
            doc_categories KEY (doc_id, category)
                SOURCE KEY (doc_id) REFERENCES acme_docs (doc_id)
                DESTINATION KEY (category) REFERENCES categories (category)
                LABEL in_category,
            doc_similarities KEY (source_doc_id, target_doc_id)
                SOURCE KEY (source_doc_id) REFERENCES acme_docs (doc_id)
                DESTINATION KEY (target_doc_id) REFERENCES acme_docs (doc_id)
                LABEL similar_to PROPERTIES (sim_score, rank_no)
        )
    """)
conn.commit()
print("Property graph ACME_GRAPH created.")

Property graph ACME_GRAPH created.


**Step 4 — query it.** The SQL below has four CTEs that mirror the algorithm:
1. `seed` — top docs by vector similarity (the starting points)
2. `sim_hops` — `GRAPH_TABLE … MATCH` across `SIMILAR_TO` edges from each seed
3. `cat_hops` — two-hop traversal `doc → category → doc` (same-category docs)
4. `scored` — blend the seed score with the edge contribution (similar edges weighted higher than same-category)

In [37]:
def graph_search(query: str, k: int = 5, seed_k: int = 5):
    """Vector-seed, then expand across SIMILAR_TO and same-category edges; blend the scores."""
    seed_k = max(seed_k, k)
    q = embed_query(query)
    sql = f"""
        WITH seed AS (
            SELECT doc_id, 1 - VECTOR_DISTANCE(embedding, :q, COSINE) AS seed_score
            FROM acme_docs
            ORDER BY seed_score DESC
            FETCH FIRST {int(seed_k)} ROWS ONLY
        ),
        seed_hits AS (
            SELECT doc_id AS candidate, seed_score, 'seed' AS rel, seed_score AS edge_score
            FROM seed
        ),
        sim_hops AS (
            SELECT gt.target_doc_id AS candidate, s.seed_score, 'similar_to' AS rel, gt.edge_score
            FROM seed s
            JOIN GRAPH_TABLE(acme_graph
                MATCH (src IS doc)-[e IS similar_to]->(dst IS doc)
                COLUMNS (src.doc_id AS source_doc_id, dst.doc_id AS target_doc_id, e.sim_score AS edge_score)
            ) gt ON gt.source_doc_id = s.doc_id
        ),
        cat_hops AS (
            SELECT gt.target_doc_id AS candidate, s.seed_score, 'same_category' AS rel, 1.0 AS edge_score
            FROM seed s
            JOIN GRAPH_TABLE(acme_graph
                MATCH (src IS doc)-[IS in_category]->(c IS category)<-[IS in_category]-(dst IS doc)
                COLUMNS (src.doc_id AS source_doc_id, dst.doc_id AS target_doc_id)
            ) gt ON gt.source_doc_id = s.doc_id
            WHERE gt.target_doc_id <> s.doc_id
        ),
        candidates AS (
            SELECT * FROM seed_hits
            UNION ALL SELECT * FROM sim_hops
            UNION ALL SELECT * FROM cat_hops
        ),
        scored AS (
            SELECT candidate AS doc_id,
                   MAX(CASE rel
                       WHEN 'seed'          THEN seed_score
                       WHEN 'similar_to'    THEN 0.70 * seed_score + 0.30 * edge_score
                       WHEN 'same_category' THEN 0.85 * seed_score + 0.15 * edge_score
                       ELSE seed_score END) AS graph_score
            FROM candidates
            GROUP BY candidate
        )
        SELECT d.doc_id, d.title, d.category, ROUND(sc.graph_score, 4) AS graph_score
        FROM scored sc JOIN acme_docs d ON d.doc_id = sc.doc_id
        ORDER BY graph_score DESC
        FETCH FIRST {int(k)} ROWS ONLY
    """
    with conn.cursor() as cur:
        cur.execute(sql, q=q)
        return cur.fetchall(), [c[0] for c in cur.description]


# Seeds on the SSO doc, then expands to its similar docs and the rest of the security/related set:
rows, cols = graph_search("How do I sign in with my company identity provider?")
show(rows, cols)

,DOC_ID,TITLE,CATEGORY,GRAPH_SCORE
0,sso,Single Sign-On (SSO),security,0.6964
1,sla,Service Level Agreement,reliability,0.6709
2,support,Support Channels & Response Times,support,0.6652
3,roles,Team Roles & Permissions,account,0.6623
4,plans,Plans & Pricing,billing,0.6126


### 2.6.6 Compare the Techniques

Run the query-driven techniques on the same query and line up their top results. Keyword returns only literal matches; vector, hybrid, and graph surface semantically or structurally related docs too.

In [ ]:
techniques = {
    "keyword": keyword_search,
    "vector": vector_search,
    "hybrid (rrf)": rrf_search,
    "graph": graph_search,
}

query = "webhooks"

print(f"Query: {query!r}\n")

for name, fn in techniques.items():
    rows, cols = fn(query, k=3)
    titles = [row[cols.index("TITLE")] for row in rows]
    print(f"{name:>13}: {titles}")

Query: 'webhooks'

      keyword: ['Webhooks']
       vector: ['Webhooks', 'Creating & Rotating API Keys', 'Exporting Your Data']
 hybrid (rrf): ['Webhooks', 'Creating & Rotating API Keys', 'Exporting Your Data']
        graph: ['Backups & Recovery', 'Webhooks', 'API Rate Limits']


## 2.7 Augment the Prompt, Then Generate

Now the RAG payoff. We wrap vector search in a single `retrieve()` helper — the **one retrieval interface the rest of the notebook reuses** (Form Factors 3 and 4 import it, so retrieval improvements propagate everywhere).

In [39]:
def retrieve(query: str, k: int = 3):
    """Unified retriever (Oracle vector search). Returns [(content, similarity), ...]."""
    q = embed_query(query)
    sql = f"""
        SELECT content, ROUND(1 - VECTOR_DISTANCE(embedding, :q, COSINE), 4) AS similarity
        FROM acme_docs
        ORDER BY similarity DESC
        FETCH FIRST {int(k)} ROWS ONLY
    """
    with conn.cursor() as cur:
        cur.execute(sql, q=q)
        return [(content, float(sim)) for content, sim in cur.fetchall()]


for content, score in retrieve("Pro plan rate limit", k=3):
    print(f"{score:.3f}  {content[:60]}...")

0.755  Acme Cloud enforces API rate limits per plan. The Free plan ...
0.619  Acme Cloud offers three plans. Free includes 1 project and c...
0.615  Acme Cloud guarantees 99.9% uptime on the Pro plan and 99.99...


Then we feed the retrieved context to Claude, instructing it to answer **only** from that context and to cite it with bracketed numbers.

### 🧩 TODO 3 — A grounded, cited RAG answer

The code in the next cell has been removed. Open **[`docs/todo3.md`](docs/todo3.md)** for a guided explanation and the solution snippet, then replace the placeholder.

Run the **`✅ TODO 3 check`** cell right after — it must pass before you continue.

In [ ]:
def rag_answer(query: str, k: int = 3) -> str:
    # 🧩 TODO 3 — retrieve, build a numbered context, then generate a cited answer. See docs/todo3.md
    raise NotImplementedError("Complete TODO 3 — open docs/todo3.md")


In [ ]:
# ✅ TODO 3 check — do not edit.
_ans = rag_answer("What is the exact API rate limit, in requests per minute, on Acme Cloud's Pro plan?")
assert isinstance(_ans, str) and _ans.strip(), "rag_answer should return a non-empty string"
assert ("1,000" in _ans) or ("1000" in _ans), "the grounded answer should contain the Pro-plan limit (1,000)"
print("✓ TODO 3 complete — the answer is grounded in your documents.")


## 2.8 Optimizing Storage with Quantization

> 💡 **Key Term — Quantization:** Storing each vector dimension in fewer bits. A FLOAT32 embedding uses 4 bytes per dimension; quantization shrinks that — trading a little accuracy for big savings in storage and query speed.

Oracle's vector index supports several precisions, selected by a **compression ratio** on the index — Oracle applies the quantization for you:

| `COMPRESSION RATIO` | FLOAT32 column becomes (in-index) | Size vs FLOAT32 |
|---|---|---|
| 2 | FLOAT16 | ~2× smaller |
| 4 | INT8 | ~4× smaller |
| 8 | (needs FLOAT64 input) | ~8× smaller |

> 💡 **Key Insight — let Oracle quantize, not you.** The documented way is a single index clause — `QUANTIZATION SCALAR COMPRESSION RATIO 4` — applied when you create the HNSW index. Oracle compresses the vectors *inside the index* (FLOAT32 → INT8 at ratio 4); the **base column stays FLOAT32**, and queries are unchanged because Oracle automatically **rescores** the top candidates against the full-precision vectors (`RESCORE FACTOR`). No client-side quantization math — you never hand-pack INT8 or binary values yourself.

In [42]:
quant_ddl = """CREATE VECTOR INDEX acme_vec_idx ON acme_docs(embedding)
    ORGANIZATION INMEMORY NEIGHBOR GRAPH
    DISTANCE COSINE WITH TARGET ACCURACY 90
    QUANTIZATION SCALAR COMPRESSION RATIO 4
    PARAMETERS (TYPE HNSW, NEIGHBORS 16, EFCONSTRUCTION 200,
                RESCORE FACTOR 2, ALGORITHM UNIFORM_QUANTIZATION)"""
print("The documented one-line way to quantize — no client-side math:\n")
print(quant_ddl, "\n")

with conn.cursor() as cur:
    try:
        cur.execute("DROP INDEX acme_vec_idx")
    except oracledb.DatabaseError:
        pass
    try:
        cur.execute(quant_ddl)
        conn.commit()
        print("✓ Quantized index created — Oracle compressed FLOAT32 → INT8 inside the index automatically.")
    except oracledb.DatabaseError as e:
        print("ℹ This Oracle build does not enable index-level scalar quantization:")
        print("   ", str(e).splitlines()[0])
        print("   (the QUANTIZATION clause is the documented approach on releases that support it).")
        print("    Recreating the standard HNSW index so the rest of the notebook keeps working...")
        cur.execute("""CREATE VECTOR INDEX acme_vec_idx ON acme_docs(embedding)
            ORGANIZATION INMEMORY NEIGHBOR GRAPH DISTANCE COSINE WITH TARGET ACCURACY 90
            PARAMETERS (TYPE HNSW, NEIGHBORS 16, EFCONSTRUCTION 200)""")
        conn.commit()

The documented one-line way to quantize — no client-side math:

CREATE VECTOR INDEX acme_vec_idx ON acme_docs(embedding)
    ORGANIZATION INMEMORY NEIGHBOR GRAPH
    DISTANCE COSINE WITH TARGET ACCURACY 90
    QUANTIZATION SCALAR COMPRESSION RATIO 4
    PARAMETERS (TYPE HNSW, NEIGHBORS 16, EFCONSTRUCTION 200,
                RESCORE FACTOR 2, ALGORITHM UNIFORM_QUANTIZATION) 

ℹ This Oracle build does not enable index-level scalar quantization:
    ORA-02158: invalid CREATE INDEX option
   (the QUANTIZATION clause is the documented approach on releases that support it).
    Recreating the standard HNSW index so the rest of the notebook keeps working...


A vector query is written **exactly the same way** whether or not the index is quantized — the compression is an index-level optimization, transparent to your SQL.

In [43]:
rows, cols = vector_search("how many requests can I send before being throttled?", k=3)
show(rows, cols)

,DOC_ID,TITLE,CATEGORY,SIMILARITY
0,rate_limits,API Rate Limits,api,0.6320
1,webhooks,Webhooks,integrations,0.5756
2,support,Support Channels & Response Times,support,0.5268


> 💡 **Key Insight — Quantization tradeoffs:** Scalar quantization to INT8 (`COMPRESSION RATIO 4`) typically preserves the top results at a quarter of the index size, because Oracle rescores the shortlist against the full-precision vectors. Higher ratios shrink the index further with more accuracy risk. The big win is that it's a *declarative* index property — you don't change your data, your inserts, or your queries.

> ### 📌 Key Takeaways — Form Factor 2
> - **RAG = retrieve, then generate.** The model answers from *your* data placed in the prompt, with citations — fixing the chatbot's data gap.
> - **Oracle AI Database** stores text, vectors, and graphs in one engine, so every retrieval technique is just SQL against one table.
> - **No retrieval strategy wins everywhere:** keyword (exact), vector (meaning), **attribute filtering** (pre/in/post — order of filter vs. search), **hybrid** (RRF, or the native `DBMS_HYBRID_VECTOR.SEARCH`), and **graph** (relationships).
> - **Quantization is a declarative index option** (`QUANTIZATION SCALAR COMPRESSION RATIO`) — Oracle compresses the vectors for you and rescores for accuracy; no client-side math.

# Form Factor 3 — The LLM-Driven Workflow

> 💡 **Key Term — Workflow:** Several LLM calls (and ordinary code) composed into a fixed, predictable pipeline. The sequence of steps is decided by **you, in code** — the model fills in each step, but it doesn't choose what happens next.

A single RAG call is great for "answer this question." But real tasks often have stages. We'll build a support pipeline that **classifies** an incoming message, **retrieves** relevant docs, **drafts** a reply, and then **reviews** that draft — revising it if it falls short.

> 💡 **Key Insight — Workflows make LLMs reliable:** Breaking a task into small, single-purpose steps with checks between them yields far more consistent results than one big do-everything prompt. Anthropic calls these patterns *prompt chaining*, *routing*, and *evaluator–optimizer* in [Building Effective Agents](https://www.anthropic.com/research/building-effective-agents).

<img src="images/form_factor_3_workflow.png" alt="Your code orchestrates the LLM" width="820">


## 3.1 Classify the Message (with structured output)

The first step routes the message. We use **structured outputs** (`output_config`) to force Claude to return clean JSON that our code can branch on — no fragile string parsing.

> 💡 **Teachable moment:** A JSON Schema with `additionalProperties: False` and every field `required` is what makes the output *machine-reliable*. The model is constrained to your shape, so `json.loads` always succeeds and your `if`/`for` logic can trust the keys.

In [45]:
import json

CLASSIFY_SCHEMA = {
    "type": "object",
    "properties": {
        "category": {
            "type": "string",
            "enum": ["billing", "technical", "account", "feature_request", "other"],
        },
        "urgency": {"type": "string", "enum": ["low", "medium", "high"]},
        "summary": {"type": "string", "description": "One-line summary of the request."},
    },
    "required": ["category", "urgency", "summary"],
    "additionalProperties": False,
}

### 🧩 TODO 4 — Reliable JSON with structured output

The code in the next cell has been removed. Open **[`docs/todo4.md`](docs/todo4.md)** for a guided explanation and the solution snippet, then replace the placeholder.

Run the **`✅ TODO 4 check`** cell right after — it must pass before you continue.

In [ ]:
def classify(message: str) -> dict:
    # 🧩 TODO 4 — classify the message into validated JSON. See docs/todo4.md
    raise NotImplementedError("Complete TODO 4 — open docs/todo4.md")


In [ ]:
# ✅ TODO 4 check — do not edit.
_info = classify("I've been double-charged for my Pro seats and need this fixed before our demo tomorrow.")
assert set(_info) >= {"category", "urgency", "summary"}, "classify must return category, urgency, summary"
assert _info["category"] == "billing", f"a double-charge is a billing issue, got {_info['category']!r}"
assert _info["urgency"] in {"low", "medium", "high"}, "urgency must be one of the allowed values"
print("✓ TODO 4 complete — structured output gives reliable JSON:", _info)


> 💡 **Why this matters in a production workload.** Forcing the model's output into a fixed JSON schema is what makes this first step *safe to build on*. In a real support pipeline this single classification drives a lot:
>
> - **Routing** — billing issues go to billing, technical to engineering.
> - **Prioritization & SLA** — an `urgency: "high"` result can page a human or jump the queue.
> - **Automation gates** — only well-classified, low-risk requests get an auto-reply; the rest are escalated to a person.
> - **Analytics** — consistent categories let you measure ticket volume and trends over time.
>
> Because the schema guarantees the shape (`additionalProperties: false`, every field required), the code that branches on `category` / `urgency` can trust those keys exist — no brittle string parsing, no surprise `KeyError` in the middle of the night. That reliability is much of the difference between a demo and a production workflow.

## 3.2 Draft a Reply

Grounded in retrieved context (reusing `retrieve()` from Form Factor 2), Claude drafts a concise, cited reply.

In [47]:
def draft_reply(message: str, context: str) -> str:
    """Step 3: draft a support reply grounded in retrieved context."""
    system = (
        "You are an Acme Cloud support agent. Write a friendly, accurate reply using ONLY "
        "the provided context. Cite facts with [n]. Keep it under 120 words."
    )
    response = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        system=system,
        messages=[{"role": "user", "content": f"Customer message:\n{message}\n\nContext:\n{context}"}],
    )
    return text_of(response)

## 3.3 Review the Draft

A second, independent call grades the draft for accuracy and tone — again returning structured JSON, so our code can act on the verdict (approve, or send feedback for a revision).

> 💡 **Teachable moment — the evaluator–optimizer pattern:** Using a *separate* LLM call to check the first one's work is one of the most effective reliability techniques. The reviewer has a single, narrow job, so it catches mistakes the drafter is blind to.

In [ ]:
REVIEW_SCHEMA = {
    "type": "object",
    "properties": {
        "approved": {"type": "boolean"},
        "feedback": {"type": "string", "description": "What to fix if not approved."},
    },
    "required": ["approved", "feedback"],
    "additionalProperties": False,
}

In [49]:

def review(message: str, draft: str, context: str) -> dict:
    """Step 4: QA the draft. Returns {approved: bool, feedback: str}."""
    system = (
        "You are a support QA reviewer. Approve the draft only if it is accurate per the "
        "context, on-topic, and professional. Otherwise set approved=false and say what to fix."
    )
    prompt = f"Customer message:\n{message}\n\nContext:\n{context}\n\nDraft reply:\n{draft}"
    response = client.messages.create(
        model=MODEL,
        max_tokens=512,
        system=system,
        messages=[{"role": "user", "content": prompt}],
        output_config={"format": {"type": "json_schema", "schema": REVIEW_SCHEMA}},
    )
    return json.loads(text_of(response))

## 3.4 Orchestrate the Pipeline

Now the control flow — and notice it's all **ordinary Python**: an `if`, a `for`, a gate. *Your code* decides the order of operations (classify → route → retrieve → draft → review → maybe revise); each LLM call does one well-defined job.

### 🧩 TODO 5 — Orchestrate the workflow pipeline

The code in the next cell has been removed. Open **[`docs/todo5.md`](docs/todo5.md)** for a guided explanation and the solution snippet, then replace the placeholder.

Run the **`✅ TODO 5 check`** cell right after — it must pass before you continue.

In [ ]:
def handle_support_message(message: str, max_revisions: int = 1) -> dict:
    # 🧩 TODO 5 — orchestrate: classify → route → retrieve → draft → review/revise. See docs/todo5.md
    raise NotImplementedError("Complete TODO 5 — open docs/todo5.md")


In [ ]:
# ✅ TODO 5 check — do not edit.
_res = handle_support_message(
    "I've been double-charged for my Pro seats and need this fixed before our board demo tomorrow."
)
assert set(_res) >= {"classification", "escalated", "reply"}, "result needs classification, escalated, reply"
assert _res["classification"]["category"] == "billing", "should classify as billing"
assert _res["escalated"] is True, "urgent billing should be escalated for a human"
assert isinstance(_res["reply"], str) and _res["reply"].strip(), "there must be a final reply"
print("✓ TODO 5 complete — your code orchestrated the whole pipeline.")


> 💡 **Key Insight — Who's in charge?** In a workflow, *the pipeline is fixed*: classify, then retrieve, then draft, then review — always in that order. That predictability is a feature. But it also means the system can't handle anything you didn't wire up in advance. When you want the system to *decide its own steps*, you've reached the next rung: the agent.
>
> For genuinely hard reasoning inside a step, enable [adaptive thinking](https://platform.claude.com/docs/en/build-with-claude/adaptive-thinking) by adding `thinking={"type": "adaptive"}` to any `messages.create` call.

> ### 📌 Key Takeaways — Form Factor 3
> - A **workflow** is multiple LLM calls wired together by *your code* into a fixed, auditable sequence.
> - **Structured outputs** (`output_config` + JSON Schema) turn model output into data your code can branch on reliably.
> - The **evaluator–optimizer** gate (draft → review → revise) is a cheap, powerful reliability boost.
> - You trade the agent's flexibility for **predictability** — the right call when the steps are known in advance.

# Form Factor 4 — The Agent

> 💡 **Key Term — Agent:** An LLM running in a loop with tools. Unlike a workflow, *the model* decides which tool to call, inspects the result, and decides what to do next — repeating until the task is done. You provide the tools and the goal; the model provides the trajectory.

We build this with the **`claude-agent-sdk`** — the same runtime that powers Claude Code. We give the agent two tools (search the docs, open a support ticket) and let it work out the rest.

> 💡 **Key Insight — Workflow vs. Agent:** In Form Factor 3 *we* wrote `classify → retrieve → draft → review`. Here we write none of that. We hand the model a goal and some tools, and it chooses the sequence itself. More flexible, less predictable — use it when the path can't be fully planned in advance.

<img src="images/four_faculties_of_an_agent.png" alt="The four faculties of an agent" width="820">


> **Note:** The `claude-agent-sdk` bundles a Node.js-based runtime and reads `ANTHROPIC_API_KEY` from your environment. Make sure Node.js is installed and the package is available (see the setup cell). The cell below imports the SDK and lets its async code run inside Jupyter.

In [51]:
import nest_asyncio
nest_asyncio.apply()  # allow the SDK's async runtime to work inside a notebook

from claude_agent_sdk import (
    query,
    tool,
    create_sdk_mcp_server,
    ClaudeAgentOptions,
    AssistantMessage,
    ResultMessage,
)
print("claude-agent-sdk imported ✓")

claude-agent-sdk imported ✓


## 4.1 Give the Agent Tools

A tool is just a Python function the model can choose to call. We define each with the `@tool` decorator. The description tells the model *when* to use it — that guidance is how the agent reasons about tool choice.

**Tool 1 — search the docs** (reuses the RAG retriever from Form Factor 2).

### 🧩 TODO 6 — Give the agent a tool

The code in the next cell has been removed. Open **[`docs/todo6.md`](docs/todo6.md)** for a guided explanation and the solution snippet, then replace the placeholder.

Run the **`✅ TODO 6 check`** cell right after — it must pass before you continue.

In [ ]:
@tool(
    "search_docs",
    "Search the Acme Cloud documentation. Use this whenever the user asks about Acme "
    "Cloud's plans, pricing, limits, or features.",
    {"query": str},
)
async def search_docs(args):
    # 🧩 TODO 6 — call the RAG retriever and return tool content. See docs/todo6.md
    raise NotImplementedError("Complete TODO 6 — open docs/todo6.md")


In [ ]:
# ✅ TODO 6 check — do not edit.
import asyncio as _asyncio
assert getattr(search_docs, "name", None) == "search_docs", "keep the @tool decorator and the 'search_docs' name"
_out = _asyncio.get_event_loop().run_until_complete(search_docs.handler({"query": "Pro plan API rate limit"}))
_txt = _out["content"][0]["text"]
assert isinstance(_txt, str) and _txt.strip(), "the tool should return the retrieved document text"
print("✓ TODO 6 complete — the agent now has a working tool it can choose to call.")


**Tool 2 — open a support ticket.** A tool can *act*, not just read. Its description tells the agent to use it only for genuine escalations.

In [53]:
@tool(
    "create_support_ticket",
    "Open a support ticket. Use this only when the user asks to escalate, or reports an "
    "urgent, unresolved problem.",
    {"summary": str, "priority": str},
)
async def create_support_ticket(args):
    ticket_id = f"ACME-{abs(hash(args['summary'])) % 10000:04d}"
    msg = f"Created ticket {ticket_id} (priority={args['priority']}): {args['summary']}"
    return {"content": [{"type": "text", "text": msg}]}

**Register the tools** on an in-process MCP server, which the agent will connect to.

> 💡 **What is an MCP server, and why wrap tools in one?** **MCP — the Model Context Protocol** — is an open standard for exposing tools, data, and prompts to LLMs through a uniform interface. Rather than every framework inventing its own way to describe and invoke a tool, the model's runtime talks to an **MCP server** that advertises a list of tools and executes them on request.
>
> The `claude-agent-sdk` speaks MCP natively, so the way you hand it custom Python functions is to put them on an MCP server. `create_sdk_mcp_server(...)` builds an **in-process** one — it runs inside this very Python process (no separate service, no network hop), and the agent calls our `search_docs` / `create_support_ticket` functions directly. Wrapping tools this way is what makes them discoverable and callable by the model — and the same tools could later be served to *any* MCP-aware client without changing the functions.

In [56]:
acme_tools = create_sdk_mcp_server(name="acme", version="1.0.0",
                                   tools=[search_docs, create_support_ticket])
print("Registered tools: search_docs, create_support_ticket")

Registered tools: search_docs, create_support_ticket


## 4.2 Run the Agent

The helper below streams the agent's messages and prints both its **tool calls** (the decisions it makes) and its **text** (what it says). Watching the tool calls is the best way to *see* the agent reasoning.

In [57]:
async def run_agent(prompt: str, options: ClaudeAgentOptions):
    """Stream an agent run, printing tool calls and assistant text as they arrive."""
    async for message in query(prompt=prompt, options=options):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if hasattr(block, "text"):                 # a TextBlock
                    print(block.text)
                elif hasattr(block, "name"):               # a ToolUseBlock
                    print(f"  🔧 {block.name}({dict(block.input)})")
        elif isinstance(message, ResultMessage):
            print("\n✓ agent finished")

We configure the agent with our two tools and a system prompt, and keep it hermetic with `setting_sources=[]` (so it ignores any local project/user settings).

**`ClaudeAgentOptions` — the agent's configuration.** Everything that shapes the run is set here and handed to `query()`:

- **`model`** — which Claude model runs the agent loop (`claude-opus-4-8`).
- **`system_prompt`** — the agent's standing instructions: its role, how it should use its tools, and the rules to follow.
- **`mcp_servers`** — a map of name → **in-process MCP server**. Our tools (`search_docs`, `create_support_ticket`) live on the `acme_tools` server we built with `create_sdk_mcp_server`; registering it here is what makes those Python functions *callable by the model*. (MCP — the Model Context Protocol — is the standard interface the SDK uses to expose tools to a model.)
- **`allowed_tools`** — an explicit allowlist of the exact tools the agent may call, written in MCP's `mcp__<server>__<tool>` form (`mcp__acme__search_docs`, `mcp__acme__create_support_ticket`). It pre-approves these tools so they run **without a permission prompt**, and just as importantly it **scopes** the agent — it will not reach for anything not on the list.
- **`setting_sources=[]`** — keeps the run hermetic by ignoring any local project/user configuration.

**Why both `mcp_servers` and `allowed_tools`?** They answer two different questions. `mcp_servers` makes a tool *exist* for the agent; `allowed_tools` decides which of the existing tools it is *permitted to use*. You need both — a registered tool that isn't allow-listed can't be called, and an allow-listed name that no registered server provides resolves to nothing.

In [58]:
support_agent_options = ClaudeAgentOptions(
    model=MODEL,
    system_prompt=(
        "You are the Acme Cloud support agent. You have exactly two tools: search_docs and "
        "create_support_ticket. Use search_docs to ground every factual claim. Only open a "
        "ticket if the user clearly needs escalation. Cite docs with [n]."
    ),
    mcp_servers={"acme": acme_tools},
    allowed_tools=["mcp__acme__search_docs", "mcp__acme__create_support_ticket"],
    setting_sources=[],
)

Now give it a request that needs *both* tools — answer a question (search) **and** escalate (ticket). Notice we never tell it the order; it decides.

In [59]:
await run_agent(
    "I'm on the Pro plan and keep hitting rate limits right before our launch tomorrow. "
    "What are my options, and can you escalate this for me?",
    support_agent_options,
)

warn: CPU lacks AVX support, strange crashes may occur. Reinstall Bun or use *-baseline build:
  https://github.com/oven-sh/bun/releases/download/bun-v1.3.14/bun-darwin-x64-baseline.zip


  🔧 ToolSearch({'query': 'select:mcp__acme__search_docs,mcp__acme__create_support_ticket', 'max_results': 5})
  🔧 mcp__acme__search_docs({'query': 'Pro plan rate limits'})
  🔧 mcp__acme__search_docs({'query': 'rate limit options upgrade increase quota'})
Here are your options, grounded in our docs:

**Why you're hitting limits:** The Pro plan caps API usage at **1,000 requests per minute** [1]. Free is 60/min, and Enterprise rate limits are negotiated per contract [1].

**Your options before launch:**

1. **Reduce your request rate** — batch calls, add client-side throttling/retry-with-backoff, or cache repeated requests to stay under 1,000/min. This is the only option that takes effect with zero waiting.
2. **Upgrade to Enterprise** for higher, contract-negotiated limits [1]. Note: standard plan upgrades take effect immediately and are pro-rated (Settings → Billing → Change Plan) [1'], but Enterprise limits are negotiated, so they aren't instant self-serve — that requires talking to o

> 💡 **Key Insight — The agent chose the path:** No `if`/`for` in our code decided to search the docs *and then* open a ticket — the model did, by reading each tool result and deciding the next move. That is the leap from workflow to agent.
>
> For a back-and-forth conversation with the same agent (memory across turns), use `ClaudeSDKClient` as an async context manager instead of `query()`.

> ### 📌 Key Takeaways — Form Factor 4
> - An **agent** is an LLM in a tool-calling loop; *the model* chooses which tools to call and when.
> - The **`claude-agent-sdk`** runs that loop for you; you supply tools (`@tool` + `create_sdk_mcp_server`), a goal, and `allowed_tools`.
> - **Tool descriptions are prompts** — they're how the model decides when each tool applies.
> - Use an agent when the path **can't be planned up front**; accept less predictability for more flexibility.

# Form Factor 5 — The Autonomous Agent

> 💡 **Key Term — Autonomous agent:** An agent equipped with tools that change the world — here, the ability to **write files and run shell commands**. It can build something durable (a script, a config, a small program), test it, and fix it, all on its own.

This is the top rung: the agent doesn't just *answer*, it *produces working automation*. We'll ask it to write a small Python script, run it, and verify the output — using the agent SDK's built-in `Write`, `Read`, `Edit`, and `Bash` tools.

> ⚠️ **Safety — this agent runs code:** With file-write and shell access plus `permission_mode="bypassPermissions"`, the agent acts without asking. That is the point of autonomy, but it means you must **scope it tightly**. We confine it to a throwaway `cwd` sandbox folder and give it a single, well-defined task. Never point an autonomous agent at a directory you care about, or at a production system, without a real permission policy.

**Set up a sandbox** and drop in a tiny dataset for the agent to build automation around.

In [41]:
import os, csv, json, pathlib

SANDBOX = pathlib.Path("./ff5_sandbox").resolve()
SANDBOX.mkdir(exist_ok=True)

rows = [
    {"id": 1, "category": "billing",   "message": "I was double charged"},
    {"id": 2, "category": "technical", "message": "API returns 500 on /v1/sync"},
    {"id": 3, "category": "billing",   "message": "How do I upgrade to Pro?"},
    {"id": 4, "category": "account",   "message": "Need to add a teammate"},
    {"id": 5, "category": "technical", "message": "Webhooks are not firing"},
]
with open(SANDBOX / "support_messages.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "category", "message"])
    writer.writeheader()
    writer.writerows(rows)

print("Sandbox ready at", SANDBOX)
print("Files:", [p.name for p in SANDBOX.iterdir()])

Sandbox ready at /Users/richmondalake/Desktop/acsxend/enablement/dbtlabs/ff5_sandbox
Files: ['support_messages.csv']


**Configure the builder agent** with the file/shell tools, pointed at the sandbox.

### 🧩 TODO 7 — Configure the autonomous builder

The code in the next cell has been removed. Open **[`docs/todo7.md`](docs/todo7.md)** for a guided explanation and the solution snippet, then replace the placeholder.

Run the **`✅ TODO 7 check`** cell right after — it must pass before you continue.

In [ ]:
builder_options = ClaudeAgentOptions()  # 🧩 TODO 7 — configure me. See docs/todo7.md


In [ ]:
# ✅ TODO 7 check — do not edit.
assert isinstance(builder_options, ClaudeAgentOptions), "builder_options must be a ClaudeAgentOptions"
assert builder_options.permission_mode == "bypassPermissions", \
    "the builder runs unattended → permission_mode='bypassPermissions'"
assert {"Read", "Write", "Edit", "Bash"}.issubset(set(builder_options.allowed_tools or [])), \
    "the builder needs Read, Write, Edit, and Bash to build & run code"
assert str(SANDBOX) in str(builder_options.cwd or ""), "keep the agent confined to the SANDBOX directory"
print("✓ TODO 7 complete — the autonomous builder is configured.")


**Turn it loose.** The agent will write a script, run it, and fix it if it errors — autonomously.

In [43]:
await run_agent(
    "In the current directory there is a file `support_messages.csv` with columns id,category,message.\n"
    "1. Write a minimal, well-documented Python script `triage_report.py` (standard library only) that "
    "reads that CSV, counts how many messages are in each category, and writes the counts to "
    "`report.json` sorted by count descending.\n"
    "2. Run it with python.\n"
    "3. If it errors, fix it and re-run until it succeeds.\n"
    "4. Confirm what you built and show the contents of report.json.",
    builder_options,
)

warn: CPU lacks AVX support, strange crashes may occur. Reinstall Bun or use *-baseline build:
  https://github.com/oven-sh/bun/releases/download/bun-v1.3.14/bun-darwin-x64-baseline.zip


I'll build this directly. Let me first check the CSV, then write the script.
  🔧 Bash({'command': 'cd "$(pwd)" && head -5 support_messages.csv && echo "---" && wc -l support_messages.csv', 'description': 'Preview CSV file and count lines'})
Now I'll write the script.
  🔧 Write({'file_path': '/Users/wweiss/Documents/scratch_workspace/triage_report.py', 'content': '#!/usr/bin/env python3\n"""Triage report generator.\n\nReads `support_messages.csv` (columns: id, category, message), counts how many\nmessages fall into each category, and writes the results to `report.json`\nsorted by count in descending order.\n\nStandard library only.\n"""\n\nimport csv\nimport json\nfrom collections import Counter\n\nINPUT_CSV = "support_messages.csv"\nOUTPUT_JSON = "report.json"\n\n\ndef main() -> None:\n    # Tally messages per category.\n    counts: Counter = Counter()\n    with open(INPUT_CSV, newline="", encoding="utf-8") as f:\n        reader = csv.DictReader(f)\n        for row in reader:\n        

## 5.1 Inspect the Artifacts

The agent didn't just describe a solution — it left real files on disk. Let's read them back.

In [44]:
print("Sandbox now contains:", [p.name for p in SANDBOX.iterdir()], "\n")

script_path = SANDBOX / "triage_report.py"
if script_path.exists():
    print("=== triage_report.py ===")
    print(script_path.read_text())

report_path = SANDBOX / "report.json"
if report_path.exists():
    print("=== report.json ===")
    print(report_path.read_text())

Sandbox now contains: ['support_messages.csv', 'report.json', 'triage_report.py'] 

=== triage_report.py ===
#!/usr/bin/env python3
"""Triage report generator.

Reads `support_messages.csv` (columns: id, category, message), counts how many
messages fall into each category, and writes the results to `report.json`
sorted by count in descending order.

Standard library only.
"""

import csv
import json
from collections import Counter

INPUT_CSV = "support_messages.csv"
OUTPUT_JSON = "report.json"


def main() -> None:
    # Tally messages per category.
    counts: Counter = Counter()
    with open(INPUT_CSV, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            category = (row.get("category") or "").strip()
            if category:  # skip rows with a missing/blank category
                counts[category] += 1

    # Sort by count descending; break ties alphabetically for stable output.
    ordered = dict(sorted(counts.items(), key=

> 💡 **Key Insight — From answering to building:** Every earlier form factor produced *text*. This one produced a *running program* that will keep working long after the agent stops. That is the defining capability of an autonomous agent — and the reason it demands the most care.

> ### 📌 Key Takeaways — Form Factor 5
> - An **autonomous agent** uses world-changing tools (`Write`, `Bash`, `Edit`) to produce **durable artifacts**, not just answers.
> - It runs a **build → run → fix** loop on its own, verifying its own work.
> - This power demands guardrails: a **sandbox `cwd`**, a tight task, and a real permission policy in production (`bypassPermissions` is for demos).
> - Top of the ladder — reach for it only when the job is to *build* something, not just respond.

# Where to Next?

You climbed all five rungs of the ladder, each adding one capability:

| # | Form Factor | New capability | Reach for it when… |
|---|-------------|----------------|--------------------|
| 1 | **Chatbot** | Generate text | The answer is in the model's general knowledge |
| 2 | **RAG** | Ground in your data | Answers must reflect *your* documents or data |
| 3 | **Workflow** | Reliable multi-step pipelines | The steps are known and you want predictability |
| 4 | **Agent** | Model-chosen tool use | The path can't be planned up front |
| 5 | **Autonomous agent** | Write & run code | The job is to *build* something, not just answer |

> 💡 **Key Insight — Climb only as high as you need:** Lower rungs are cheaper, faster, and more predictable. Start at the simplest form factor that solves your problem, and move up only when a real limitation forces you to.

**Keep learning:**

- **[Building Effective Agents](https://www.anthropic.com/research/building-effective-agents)** — Anthropic's guide to workflows vs. agents
- **[Claude Agent SDK docs](https://code.claude.com/docs/en/agent-sdk/overview)** — build custom agents (Form Factors 4–5)
- **[Claude API docs](https://platform.claude.com/docs/en/api/overview)** — the Messages API behind Form Factors 1–3
- **[Structured outputs](https://platform.claude.com/docs/en/build-with-claude/structured-outputs)** — the JSON-schema technique from the workflow
- **[Oracle AI Vector Search](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/)** — vectors, hybrid search, and quantization in Oracle